[⬅ Back to the examples](https://anywidgetinstruments.github.io/anywidget-instruments-industrial/examples/)

# Real-time signal acquisition display

A simulated two-channel acquisition card delivers blocks of 50 samples every
50 ms (1 kHz per channel). The panel shows the raw signals on a strip chart,
a spectrogram (FFT of each block) on an intensity chart, the RMS level on a
VU meter and an estimate of the dominant frequency on a seven-segment display.

> **The code is hidden.** To see a cell's code, click the `⋯` bar above its result; to see all of it, choose **View ▸ Expand All Code**.

**References:** SI prefixes (engineering units). These standards inspired
the widgets; the library does not claim conformity with them ([Standards and
references](https://anywidgetinstruments.github.io/anywidget-instruments-industrial/standards/)).

In [ ]:
import os
import threading
import time

import numpy as np

import anywidget_instruments as ai

# How long the live simulation runs (seconds). The documentation build and the
# test-suite set AWI_EXAMPLE_SECONDS to run the examples briefly.
RUN_SECONDS = float(os.environ.get("AWI_EXAMPLE_SECONDS", "600"))

FS = 1000.0  # sample rate, Hz
BLOCK = 50  # samples per block

In [ ]:
ai.theme_switch()  # light / system / dark theme (STYLE-008)

In [ ]:
freq = ai.Knob(
    12, min=1, max=200, step=1, unit="Hz", label="Frequency", scale="log", size=(120, 120)
)
amp = ai.Knob(1.0, min=0, max=2, step=0.05, unit="V", label="Amplitude", size=(120, 120))
noise = ai.Knob(0.1, min=0, max=1, step=0.01, unit="V", label="Noise", size=(120, 120))
acquire = ai.ToggleSwitch(True, label="ACQUIRE")
vu = ai.VUMeter(
    0,
    min=0,
    max=2,
    unit="V",
    label="RMS",
    peak_hold=True,
    peak_decay=2,
    hi=1.2,
    hihi=1.6,
    size=(50, 200),
)
f_est = ai.SevenSegment(0, digits=4, decimals=0, label="Peak Hz", size=(160, 64))
led = ai.LED(label="Clipping", on_color="#dc2626")
scope = ai.WaveformChart(
    n_traces=2,
    history=2000,
    dt=1 / FS,
    x_unit="s",
    unit="V",
    y_min=-2.5,
    y_max=2.5,
    size=(620, 220),
    label="Channels",
    traces=[{"name": "CH0 sine"}, {"name": "CH1 square"}],
)
spectrum = ai.IntensityChart(
    n_bins=BLOCK * 2 // 2,
    history=200,
    dt=BLOCK / FS,
    y_min=0,
    y_max=FS / 2,
    x_unit="s",
    unit="dBV",
    autoscale_z=False,
    z_min=-60,
    z_max=0,
    colormap="inferno",
    size=(620, 220),
    label="Spectrogram CH0",
)

ai.Panel(
    [
        ai.Panel([freq, amp, noise, acquire, vu, ai.Panel([f_est, led], columns=1)], columns=6),
        scope,
        spectrum,
    ],
    columns=1,
)

In [ ]:
def acquire_loop(duration=RUN_SECONDS):
    k = 0
    window = np.hanning(2 * BLOCK)
    history = np.zeros(2 * BLOCK)
    t_end = time.time() + duration
    while time.time() < t_end:
        time.sleep(BLOCK / FS)
        if not acquire.value:
            continue
        t = (k * BLOCK + np.arange(BLOCK)) / FS
        k += 1
        ch0 = amp.value * np.sin(2 * np.pi * freq.value * t)
        ch0 += noise.value * np.random.randn(BLOCK)
        ch1 = 0.8 * np.sign(np.sin(2 * np.pi * 2 * t))
        history = np.concatenate([history[BLOCK:], ch0])
        spec = np.abs(np.fft.rfft(history * window))[:BLOCK] / BLOCK
        with ai.batch():
            scope.append(np.column_stack([ch0, ch1]))
            spectrum.append(20 * np.log10(spec + 1e-6))
            vu.value = float(np.sqrt(np.mean(ch0**2)))
            f_est.value = float(np.argmax(spec[1:]) + 1) * FS / (2 * BLOCK)
            led.value = bool(np.max(np.abs(ch0)) > 2.0)


threading.Thread(target=acquire_loop, daemon=True).start()

**Things to try**

* Sweep the frequency knob and watch the line move on the spectrogram.
* Raise the noise: the RMS level crosses the HI / HIHI segments.
* Pause the scope (`scope.paused = True`), then zoom and add cursors to
  measure the period; export the data as CSV from the toolbar.